# NeuroRAG Ablation Study on MMLU

This notebook allows you to disable (ablate) almost any component of the NeuroRAG pipeline and evaluate the effect on MMLU accuracy across multiple medical and scientific domains.

In [1]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import re
import json
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

from neurorag.neurorag import NeuroRAG
from datasets import load_dataset

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Disable warnings

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

In [3]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

KeyboardInterrupt: Interrupted by user

## Ablation Config
Set any component to False to disable it in the pipeline.

In [ ]:
ablation_config: dict[str, bool] = {
    'step_back': True,
    'query_rewriting': True,
    'decomposition': True,
    'hyde': True,
    'vector_store': True,
    'pubmed': True,
    'arxiv': True,
    'ncbi_protein': True,
    'ncbi_gene': True,
    'biorxiv': True,
    'medrxiv': True,
    'document_grading': True,
    'hallucination_grading': True,
    'answer_grading': True,
    'web_search': True,
}


## NeuroRAG Wrapper for Ablation
This class disables components according to the config above.

In [ ]:
class NeuroRAGAblation(NeuroRAG):
    def __init__(self, ablation_config, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ablation_config = ablation_config

    def generate_step_back_query_node(self, state):
        if not self.ablation_config.get('step_back', True):
            return {'step_back_query': state['query']}
        return super().generate_step_back_query_node(state)

    def generate_rewritten_query_node(self, state):
        if not self.ablation_config.get('query_rewriting', True):
            return {'rewritten_query': state['query']}
        return super().generate_rewritten_query_node(state)

    def generate_subqueries_node(self, state):
        if not self.ablation_config.get('decomposition', True):
            return {'subqueries': []}
        return super().generate_subqueries_node(state)

    def generate_hyde_documents_node(self, state):
        if not self.ablation_config.get('hyde', True):
            return {'generated_documents': [state['query']]}
        return super().generate_hyde_documents_node(state)

    def vector_store_retriever_node(self, state):
        if not self.ablation_config.get('vector_store', True):
            return {'documents': []}
        return super().vector_store_retriever_node(state)

    def pub_med_retriever_node(self, state):
        if not self.ablation_config.get('pubmed', True):
            return {'documents': []}
        return super().pub_med_retriever_node(state)

    def arxiv_retriever_node(self, state):
        if not self.ablation_config.get('arxiv', True):
            return {'documents': []}
        return super().arxiv_retriever_node(state)

    def ncbi_protein_db_retriever_node(self, state):
        if not self.ablation_config.get('ncbi_protein', True):
            return {'documents': []}
        return super().ncbi_protein_db_retriever_node(state)

    def ncbi_gene_db_retriever_node(self, state):
        if not self.ablation_config.get('ncbi_gene', True):
            return {'documents': []}
        return super().ncbi_gene_db_retriever_node(state)

    def biorxiv_retriever_node(self, state):
        if not self.ablation_config.get('biorxiv', True):
            return {'documents': []}
        return super().biorxiv_retriever_node(state)

    def medrxiv_retriever_node(self, state):
        if not self.ablation_config.get('medrxiv', True):
            return {'documents': []}
        return super().medrxiv_retriever_node(state)

    def grade_documents_node(self, state):
        if not self.ablation_config.get('document_grading', True):
            # Bypass grading, just pass all documents through
            return {'documents': state['documents'], 'web_search': False}
        return super().grade_documents_node(state)

    def grade_generation_node(self, state):
        generations_number = state['generations_number']

        if generations_number >= self.max_retries:
            return 'useful'

        if not self.ablation_config.get('hallucination_grading', True) and not self.ablation_config.get('answer_grading', True):
            return 'useful'
        if not self.ablation_config.get('hallucination_grading', True):
            # Only answer grading
            query = state['query']
            generation = state['generation']
            try:
                grade = self.answer_grade_chain.invoke(query, generation)
            except Exception:
                grade = 'no'
            return 'useful' if grade == 'yes' else 'not useful'
        if not self.ablation_config.get('answer_grading', True):
            # Only hallucination grading
            documents = state['documents']
            generation = state['generation']
            try:
                context = (
                    '\n\n' + '\n\n'.join(map(lambda doc: doc.page_content, documents)) + '\n\n'
                )
                grade = self.hallucinations_chain.invoke(generation, context)
            except Exception:
                grade = 'no'
            return 'useful' if grade == 'yes' else 'not useful'
        return super().grade_generation_node(state)

    def web_search_node(self, state):
        if not self.ablation_config.get('web_search', True):
            return {'documents': [], 'web_results': []}
        return super().web_search_node(state)


## Setup evaluation

### Load cache

In [ ]:
try:
  with open('cache.json', 'r') as file:
    cache = json.load(file)
except FileNotFoundError:
  cache = {}

### Setup MMLU evaluation

In [ ]:
def extract_json(response):
  json_pattern = r'\{.*?\}'
  match = re.search(json_pattern, response, re.DOTALL)

  if match:
    return match.group().strip().replace('\\\\', '\\')

  return response

class RAGSchema(BaseModel):
  correct_answer: str = Field(description='Based on the question and the provided context, choose the most accurate letter among [A, B, C, D].')

rag_parser = PydanticOutputParser(pydantic_object=RAGSchema)

rag_template = """
You are a knowledgeable assistant with expertise in multiple domains.
Read the following question and context carefully.

1. Use the context and your domain knowledge to determine the correct answer.
2. Do any necessary reasoning internally—do not include your chain of thought in the output.
3. Provide only the one-letter answer (from [A, B, C, D]) in valid JSON format.

{format_instructions}

Question:
{query}

Context (review carefully):
{context}

Possible answers:
A. {a}
B. {b}
C. {c}
D. {d}
"""
prompt = PromptTemplate(
  template=rag_template,
  input_variables=['query', 'a', 'b', 'c', 'd', 'context'],
  partial_variables={'format_instructions': rag_parser.get_format_instructions()},
)

letter_to_number = {'a': 0, 'b': 1, 'c': 2, 'd': 3}

,question,answer
0,SSPE. My son is 33years of age and did not hav...,Subacute sclerosing panencephalitis: Subacute ...
1,Homozygout MTHFR A1298C Health Issues and long...,MTHFR gene variant (Inheritance): Because each...
2,What is Stroke?,Stroke: A stroke occurs when the blood supply ...
3,What causes Stroke?,Ischemic Stroke (Summary): Summary A stroke is...
4,What are the symptoms of Stroke?,What are the symptoms of Stroke?: The signs an...
5,What are the treatments of Stroke?,Stroke (Treatment): A stroke is a medical emer...
6,What is Dementia?,Dementia (WHAT IS DEMENTIA?): Dementia is the ...
7,What causes Dementia?,What causes Dementia?: Dementia usually occurs...
8,What are the symptoms of Dementia?,Dementia (Symptoms): Dementia symptoms include...
9,How to diagnose Dementia?,Dementia (Diagnosis): Diagnosing dementia and ...


### MMLU Evaluation function

In [ ]:
import time

def invoke_with_retries(app, question, k = 10, sleep: int = 5):
    for i in range(k):
        try:
            return app.invoke(question)
        except Exception as e:
            if i == k - 1:
                raise e
            else:
                print(e)
            time.sleep(sleep * i)

In [ ]:
def serialize_response(response: dict) -> dict:
  serialized_response = response.copy()
  serialized_response['documents'] = [doc.dict() for doc in response['documents']]
  serialized_response['web_results'] = [doc.dict() for doc in response['web_results']]
  return serialized_response

In [ ]:
def eval_rag_mmlu(app, experiment_name, mmlu_subset: str) -> float:
    dataset = load_dataset('cais/mmlu', mmlu_subset)
    test_df = dataset['test'].to_pandas()

    correct_answers_count = 0

    if 'ablation_study' not in cache:
        cache['ablation_study'] = {}
    if experiment_name not in cache['ablation_study']:
        cache['ablation_study'][experiment_name] = {}
    if mmlu_subset not in cache['ablation_study'][experiment_name]:
        cache['ablation_study'][experiment_name][mmlu_subset] = {}

    for index, row in tqdm(list(test_df.iterrows()), desc=f'{experiment_name} - {mmlu_subset}'):
        question = row['question']
        choices = row['choices']
        correct_answer = row['answer']

        if question not in cache['ablation_study'][experiment_name][mmlu_subset]:
            prompt_with_choices = prompt.partial(
                a=choices[0],
                b=choices[1],
                c=choices[2],
                d=choices[3],
            )
            app.generation_prompt = prompt_with_choices
            
            try:
                response = invoke_with_retries(app, question)
                llm_answer_letter = response['generation'].strip().lower()[0]
                cache['ablation_study'][experiment_name][mmlu_subset][question] = {
                    'answer': llm_answer_letter,
                    'full_response': serialize_response(response)
                }
                with open('cache.json', 'w') as file:
                    json.dump(cache, file)
            except Exception as e:
                print(f"Error processing question {index}: {e}")
                continue

        llm_answer_letter = cache['ablation_study'][experiment_name][mmlu_subset][question]['answer']

        if llm_answer_letter not in letter_to_number:
            print(f'Question {index + 1}, correct answer: {correct_answer}, model answer letter: {llm_answer_letter}')
            continue

        llm_answer_num = letter_to_number[llm_answer_letter]

        if llm_answer_num == correct_answer:
            correct_answers_count += 1
        else:
            print(f'Question {index + 1}, correct answer: {correct_answer}, model answer: {llm_answer_num}')

    return correct_answers_count / len(test_df)

## Run MMLU Ablation Study
Try a few ablation settings and compare results across multiple MMLU domains.

In [ ]:
# Define MMLU subsets to evaluate
mmlu_subsets = [
    'anatomy',
    'clinical_knowledge', 
    'college_biology',
    'college_medicine',
    'medical_genetics',
    'professional_medicine',
    'professional_psychology',
    'high_school_psychology',
    'college_chemistry',
    'high_school_chemistry',
]

# Define ablation settings to test
ablation_settings = [
    ('All enabled', ablation_config.copy()),
    ('No step_back', {**ablation_config, 'step_back': False}),
    ('No query_rewriting', {**ablation_config, 'query_rewriting': False}),
    ('No decomposition', {**ablation_config, 'decomposition': False}),
    ('No hyde', {**ablation_config, 'hyde': False}),
    ('No document_grading', {**ablation_config, 'document_grading': False}),
    ('No hallucination_grading', {**ablation_config, 'hallucination_grading': False}),
    ('No answer_grading', {**ablation_config, 'answer_grading': False}),
    ('No web_search', {**ablation_config, 'web_search': False}),
    ('No retrievers', {**ablation_config, 'vector_store': False, 'pubmed': False, 'arxiv': False, 'ncbi_gene': False, 'ncbi_protein': False, 'biorxiv': False, 'medrxiv': False}),
]

# Run ablation study
results = []
for name, config in ablation_settings:
    print(f'\n=== Running: {name} ===')
    app = NeuroRAGAblation(config, model='llama3.1', debug=False)
    app.compile()
    
    experiment_results = {'experiment_name': name}
    
    for subset in mmlu_subsets:
        print(f'Evaluating {subset}...')
        accuracy = eval_rag_mmlu(app, name, subset)
        experiment_results[f'{subset}_accuracy'] = accuracy
        print(f'{subset}: {accuracy:.3f}')
    
    # Calculate average accuracy across all subsets
    subset_accuracies = [experiment_results[f'{subset}_accuracy'] for subset in mmlu_subsets]
    experiment_results['average_accuracy'] = sum(subset_accuracies) / len(subset_accuracies)
    
    results.append(experiment_results)
    print(f'{name} - Average accuracy: {experiment_results["average_accuracy"]:.3f}')

# Create results DataFrame
results_df = pd.DataFrame(results)
print('\n=== Results Summary ===')
print(results_df)

In [ ]:
# Save results to file
results_df.to_csv('mmlu_ablation_results.csv', index=False)
print("Results saved to mmlu_ablation_results.csv")

# Display results in a more readable format
print("\n=== Detailed Results ===")
for _, row in results_df.iterrows():
    print(f"\n{row['experiment_name']}:")
    print(f"  Average Accuracy: {row['average_accuracy']:.3f}")
    for subset in mmlu_subsets:
        print(f"  {subset}: {row[f'{subset}_accuracy']:.3f}")


In [ ]:
# Create visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the plotting style
plt.style.use('default')
sns.set_palette("husl")

# Create a heatmap of accuracies across experiments and subsets
subset_columns = [f'{subset}_accuracy' for subset in mmlu_subsets]
heatmap_data = results_df.set_index('experiment_name')[subset_columns]

plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            cbar_kws={'label': 'Accuracy'}, linewidths=0.5)
plt.title('MMLU Ablation Study Results - Accuracy by Domain and Experiment')
plt.xlabel('MMLU Subsets')
plt.ylabel('Experiments')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Create a bar plot of average accuracies
plt.figure(figsize=(12, 6))
plt.bar(range(len(results_df)), results_df['average_accuracy'])
plt.xlabel('Experiments')
plt.ylabel('Average Accuracy')
plt.title('Average MMLU Accuracy Across All Domains')
plt.xticks(range(len(results_df)), results_df['experiment_name'], rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
